In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from pathlib import Path
import os
import warnings
warnings.filterwarnings("ignore")

RAW       = Path(os.getcwd())
PROCESSED = RAW / "processed"

df_nav   = pd.read_csv(PROCESSED / "clean_nav.csv",         parse_dates=["date"])
df_fund  = pd.read_csv(PROCESSED / "clean_fund_master.csv")
df_perf  = pd.read_csv(PROCESSED / "clean_performance.csv")
df_bench = pd.read_csv(PROCESSED / "clean_benchmark.csv",   parse_dates=["date"])

print("All loaded!")
print("NAV shape:", df_nav.shape)
print("Benchmark shape:", df_bench.shape)

All loaded!
NAV shape: (46000, 4)
Benchmark shape: (8050, 3)


In [2]:
df_nav = df_nav.sort_values(["amfi_code", "date"]).reset_index(drop=True)

df_nav["daily_return"] = (
    df_nav.groupby("amfi_code")["nav"]
          .pct_change()
)

# Annualised return per fund
annual_returns = (
    df_nav.groupby("amfi_code")["daily_return"]
          .apply(lambda x: (1 + x.dropna()).prod() ** (252 / len(x.dropna())) - 1)
          .reset_index()
)
annual_returns.columns = ["amfi_code", "annualised_return"]
annual_returns["annualised_return_pct"] = (annual_returns["annualised_return"] * 100).round(2)

# Merge with fund names
annual_returns = annual_returns.merge(
    df_fund[["amfi_code", "scheme_name", "fund_house", "category"]],
    on="amfi_code", how="left"
)

print(annual_returns[["scheme_name", "annualised_return_pct"]].sort_values(
    "annualised_return_pct", ascending=False).to_string(index=False))

annual_returns.to_csv(PROCESSED / "returns_computed.csv", index=False)
print("\nSaved returns_computed.csv")

                                          scheme_name  annualised_return_pct
             ICICI Pru Midcap Fund - Regular - Growth                  31.51
           SBI Small Cap Fund - Regular Plan - Growth                  31.13
                DSP Small Cap Fund - Regular - Growth                  31.00
        Mirae Asset Tax Saver Fund - Regular - Growth                  30.67
        Mirae Asset Large Cap Fund - Regular - Growth                  29.74
               Kotak Flexicap Fund - Regular - Growth                  29.68
   HDFC Mid-Cap Opportunities Fund - Regular - Growth                  28.93
                   DSP Midcap Fund - Regular - Growth                  28.41
                  Axis Midcap Fund - Regular - Growth                  27.10
            SBI Bluechip Fund - Regular Plan - Growth                  24.80
       Nippon India Large Cap Fund - Regular - Growth                  23.12
        ABSL Frontline Equity Fund - Regular - Growth                  22.63

In [3]:
def compute_cagr(df, amfi_code, years):
    df_fund = df[df["amfi_code"] == amfi_code].sort_values("date")
    if df_fund.empty:
        return None
    nav_end   = df_fund["nav"].iloc[-1]
    end_date  = df_fund["date"].iloc[-1]
    start_date = end_date - pd.DateOffset(years=years)
    df_start  = df_fund[df_fund["date"] >= start_date]
    if df_start.empty:
        return None
    nav_start = df_start["nav"].iloc[0]
    cagr = (nav_end / nav_start) ** (1 / years) - 1
    return round(cagr * 100, 2)

all_codes = df_nav["amfi_code"].unique()
cagr_rows = []

for code in all_codes:
    row = {"amfi_code": code}
    for yr in [1, 3, 5]:
        row[f"cagr_{yr}yr_pct"] = compute_cagr(df_nav, code, yr)
    cagr_rows.append(row)

df_cagr = pd.DataFrame(cagr_rows)
df_cagr = df_cagr.merge(
    df_fund[["amfi_code", "scheme_name", "fund_house", "category"]],
    on="amfi_code", how="left"
)

print(df_cagr[["scheme_name", "cagr_1yr_pct", "cagr_3yr_pct", "cagr_5yr_pct"]]
      .sort_values("cagr_3yr_pct", ascending=False).to_string(index=False))

df_cagr.to_csv(PROCESSED / "cagr_report.csv", index=False)
print("\nSaved cagr_report.csv")

                                          scheme_name  cagr_1yr_pct  cagr_3yr_pct  cagr_5yr_pct
                  Axis Midcap Fund - Regular - Growth         22.26         35.11         24.45
        Mirae Asset Large Cap Fund - Regular - Growth         20.36         34.00         26.80
            ICICI Pru Bluechip Fund - Direct - Growth         13.06         32.49         20.23
   HDFC Mid-Cap Opportunities Fund - Regular - Growth         53.23         32.44         26.07
             ICICI Pru Midcap Fund - Regular - Growth         29.60         31.78         28.38
            SBI Bluechip Fund - Regular Plan - Growth         60.44         30.46         22.38
               Kotak Flexicap Fund - Regular - Growth         26.66         29.58         26.74
        Mirae Asset Tax Saver Fund - Regular - Growth         39.75         29.18         27.63
        ABSL Frontline Equity Fund - Regular - Growth         47.92         28.97         20.44
                DSP Small Cap Fund - Reg

In [4]:
RISK_FREE_RATE = 0.065  # RBI repo rate proxy 6.5%
rf_daily       = RISK_FREE_RATE / 252

sharpe_rows = []

for code, group in df_nav.groupby("amfi_code"):
    returns = group["daily_return"].dropna()
    if len(returns) < 30:
        continue
    excess  = returns - rf_daily
    sharpe  = (excess.mean() / returns.std()) * np.sqrt(252)
    sharpe_rows.append({"amfi_code": code, "sharpe_ratio_computed": round(sharpe, 3)})

df_sharpe = pd.DataFrame(sharpe_rows)
df_sharpe = df_sharpe.merge(
    df_fund[["amfi_code", "scheme_name", "category"]],
    on="amfi_code", how="left"
)

print(df_sharpe.sort_values("sharpe_ratio_computed", ascending=False).to_string(index=False))
df_sharpe.to_csv(PROCESSED / "sharpe_values.csv", index=False)
print("\nSaved sharpe_values.csv")

 amfi_code  sharpe_ratio_computed                                           scheme_name category
    148567                  1.448         Mirae Asset Large Cap Fund - Regular - Growth   Equity
    120843                  1.307                Kotak Flexicap Fund - Regular - Growth   Equity
    148569                  1.235         Mirae Asset Tax Saver Fund - Regular - Growth   Equity
    119551                  1.208             SBI Bluechip Fund - Regular Plan - Growth   Equity
    120505                  1.180              ICICI Pru Midcap Fund - Regular - Growth   Equity
    149323                  1.132                    DSP Midcap Fund - Regular - Growth   Equity
    100033                  1.094    HDFC Mid-Cap Opportunities Fund - Regular - Growth   Equity
    118632                  1.082        Nippon India Large Cap Fund - Regular - Growth   Equity
    101206                  1.027         ABSL Frontline Equity Fund - Regular - Growth   Equity
    120504                  1.

In [5]:
sortino_rows = []

for code, group in df_nav.groupby("amfi_code"):
    returns = group["daily_return"].dropna()
    if len(returns) < 30:
        continue
    excess        = returns - rf_daily
    downside      = returns[returns < 0]
    downside_std  = downside.std() * np.sqrt(252)
    ann_excess    = excess.mean() * 252
    sortino       = ann_excess / downside_std if downside_std != 0 else np.nan
    sortino_rows.append({"amfi_code": code, "sortino_ratio": round(sortino, 3)})

df_sortino = pd.DataFrame(sortino_rows)
df_sortino = df_sortino.merge(
    df_fund[["amfi_code", "scheme_name", "category"]],
    on="amfi_code", how="left"
)

print(df_sortino.sort_values("sortino_ratio", ascending=False).to_string(index=False))
df_sortino.to_csv(PROCESSED / "sortino_values.csv", index=False)
print("\nSaved sortino_values.csv")

 amfi_code  sortino_ratio                                           scheme_name category
    148567          2.386         Mirae Asset Large Cap Fund - Regular - Growth   Equity
    120843          2.364                Kotak Flexicap Fund - Regular - Growth   Equity
    148569          2.147         Mirae Asset Tax Saver Fund - Regular - Growth   Equity
    119551          2.140             SBI Bluechip Fund - Regular Plan - Growth   Equity
    120505          2.029              ICICI Pru Midcap Fund - Regular - Growth   Equity
    149323          1.875                    DSP Midcap Fund - Regular - Growth   Equity
    118632          1.850        Nippon India Large Cap Fund - Regular - Growth   Equity
    100033          1.829    HDFC Mid-Cap Opportunities Fund - Regular - Growth   Equity
    120504          1.805             ICICI Pru Bluechip Fund - Direct - Growth   Equity
    101206          1.800         ABSL Frontline Equity Fund - Regular - Growth   Equity
    119094          1

In [6]:
# Get Nifty 100 benchmark returns
df_nifty = df_bench[df_bench["index_name"] == "NIFTY100"].sort_values("date").copy()
df_nifty["bench_return"] = df_nifty["close_value"].pct_change()
df_nifty = df_nifty[["date", "bench_return"]].dropna()

alpha_beta_rows = []

for code, group in df_nav.groupby("amfi_code"):
    group = group.sort_values("date")
    merged = group.merge(df_nifty, on="date", how="inner")
    merged = merged.dropna(subset=["daily_return", "bench_return"])

    if len(merged) < 30:
        continue

    slope, intercept, r, p, se = stats.linregress(
        merged["bench_return"], merged["daily_return"]
    )

    beta  = round(slope, 3)
    alpha = round(intercept * 252 * 100, 3)   # annualised alpha in %

    alpha_beta_rows.append({
        "amfi_code": code,
        "alpha_pct": alpha,
        "beta":      beta,
        "r_squared": round(r**2, 3)
    })

df_ab = pd.DataFrame(alpha_beta_rows)
df_ab = df_ab.merge(
    df_fund[["amfi_code", "scheme_name", "category", "fund_house"]],
    on="amfi_code", how="left"
)

print(df_ab.sort_values("alpha_pct", ascending=False).to_string(index=False))
df_ab.to_csv(PROCESSED / "alpha_beta.csv", index=False)
print("\nSaved alpha_beta.csv")

 amfi_code  alpha_pct   beta  r_squared                                           scheme_name category               fund_house
    119598     30.337 -0.023      0.000            SBI Small Cap Fund - Regular Plan - Growth   Equity          SBI Mutual Fund
    149324     30.058  0.011      0.000                 DSP Small Cap Fund - Regular - Growth   Equity          DSP Mutual Fund
    120505     29.264  0.001      0.000              ICICI Pru Midcap Fund - Regular - Growth   Equity      ICICI Prudential MF
    148569     28.270  0.018      0.000         Mirae Asset Tax Saver Fund - Regular - Growth   Equity           Mirae Asset MF
    120843     27.330 -0.023      0.000                Kotak Flexicap Fund - Regular - Growth   Equity        Kotak Mahindra MF
    100033     27.195  0.005      0.000    HDFC Mid-Cap Opportunities Fund - Regular - Growth   Equity         HDFC Mutual Fund
    148567     26.984  0.024      0.000         Mirae Asset Large Cap Fund - Regular - Growth   Equity  